In [1]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
%matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [3]:
data = {}
nas_dir = C.device_paths()[0]
output_dir = os.path.join(nas_dir, "Simon", "randomthings")
Logger().init_logger(None, None, logging_level="WARNING")



In [4]:
animal_ids = [10]
paradigm_ids = [1100]
session_ids = None

In [5]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
session_names = fullfnames2snames(session_dirs)

SINGLE UNIT ACTIVATIONS
===================================

SINGLE UNIT ACTIVATIONS
===================================

SINGLE UNIT ACTIVATIONS
===================================

SINGLE UNIT ACTIVATIONS
===================================

In [6]:
# firing rates
fr_raw = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr = fr_raw.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0,1,3, ))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])

# for concat projection
fr = fr.fillna(0)
fr_z = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
fr_z

AttributeError: 'NoneType' object has no attribute 'drop'

In [ ]:
# ==============================================================
# === COMPUTATION UTILITIES ====================================
# ==============================================================
from scipy import stats

def compute_distribution_params(unit_hz):
    """
    Compute firing rate statistics and PMFs for Poisson and Negative Binomial.
    Returns dict with mean, std, variance, PMFs, etc.
    """
    m = unit_hz.mean()
    std = unit_hz.std()
    var = std ** 2

    # Convert firing rates to discrete counts
    discrete_counts = np.round(unit_hz.values).astype(int)
    discrete_counts = discrete_counts[discrete_counts >= 0]
    lam = discrete_counts.mean()

    # PMFs
    x = np.arange(0, max(10, discrete_counts.max()) + 1)
    pois_pmf = stats.poisson.pmf(x, lam)

    # Negative binomial params
    p = m / var if var > m else 0.999  # prevent division by zero
    r = m * p / (1 - p)
    nb_pmf = stats.nbinom.pmf(x, r, p)

    return {
        "mean": m,
        "std": std,
        "var": var,
        "lam": lam,
        "x": x,
        "pois_pmf": pois_pmf,
        "nb_pmf": nb_pmf,
    }
# import numpy as np
# import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


# ==============================================================
# === PLOTTING UTILITIES =======================================
# ==============================================================

def plot_firing_rate_trace(ax, unit_hz, session_t_start, fr_index, neuron_i):
    """Plot main firing rate trace with session demarcations."""
    ax.plot(fr_index.get_level_values('global_t') / (1e6 * 60), unit_hz, alpha=0.7)
    for i in range(len(session_t_start)):
        s_id, _, global_t_start = session_t_start[i]
        global_t_end = (
            session_t_start[i + 1][2]
            if i + 1 < len(session_t_start)
            else fr_index.get_level_values('global_t').max()
        )
        avg_fr = unit_hz[
            (fr_index.get_level_values('global_t') >= global_t_start)
            & (fr_index.get_level_values('global_t') < global_t_end)
        ].mean()
        s_str = f"S{s_id:02}\nAvg {avg_fr:.1f}Hz"
        ax.axvline(global_t_start / (1e6 * 60), color='gray', linestyle='--', alpha=0.5)
        ax.text(global_t_start / (1e6 * 60) + 0.1, ax.get_ylim()[1] * 0.9, s_str,
                verticalalignment='top', color='gray', alpha=0.7)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f'Neuron {neuron_i}')
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('Firing Rate (Hz)')
    ax.yaxis.set_major_locator(MultipleLocator(25))
    ax.grid(axis='y', linestyle='--', alpha=0.5)


def plot_histograms(ax1, ax2, unit_hz, x, nb_pmf, pois_pmf):
    """Plot histogram + Poisson and NegBinom fits."""
    # First histogram (log x-scale)
    ax1.hist(unit_hz, bins=20, orientation='horizontal', color='gray', alpha=0.7, density=True)
    ax1.plot(nb_pmf, x, 'b--', lw=2)
    ax1.plot(np.clip(pois_pmf, 1e-22, None), x, 'r--', lw=2)
    ax1.set_xscale('log')
    ax1.set_xlabel('Count')
    ax1.set_ylabel('Firing Rate (Hz)')
    ax1.yaxis.set_major_locator(MultipleLocator(25))
    ax1.grid(axis='y', linestyle='--', alpha=0.5)

    # Second histogram (linear x-scale)
    bins = ax2.hist(unit_hz, bins=20, orientation='horizontal', color='gray', alpha=0.7, density=True)[0]
    ax2.plot(nb_pmf, x, 'b--', lw=2, label='NegBinom')
    ax2.plot(pois_pmf, x, 'r--', lw=2, label='Poisson')
    ax2.set_xlabel('Density')
    ax2.set_ylabel('Firing Rate (Hz)')
    ax2.legend()
    ax2.yaxis.set_major_locator(MultipleLocator(25))
    ax2.set_xlim(0, np.max(bins) * 1.2)
    ax2.grid(axis='y', linestyle='--', alpha=0.5)


# ==============================================================
# === MAIN WRAPPER =============================================
# ==============================================================

def plot_neuron_summary(fr, session_t_start, neuron_indices=None):
    """
    For each neuron, plot firing rate trace and histogram with Poisson/NB fits.
    """
    plt.close('all')
    if neuron_indices is None:
        neuron_indices = range(fr.shape[1])

    all_params = []
    for neuron_i in neuron_indices:
        print(neuron_i)
        fig, ax = plt.subplots(
            ncols=3, nrows=1, figsize=(40, 3), sharey=False,
            tight_layout=True, width_ratios=(25, 1, 1)
        )
        fig.subplots_adjust(left=0.05, right=0.95)

        unit_hz = fr.iloc[:, neuron_i]
        params = compute_distribution_params(unit_hz)
        all_params.append(params)

        plot_firing_rate_trace(ax[0], unit_hz, session_t_start, fr.index, neuron_i)
        plot_histograms(ax[1], ax[2], unit_hz, params["x"], params["nb_pmf"], params["pois_pmf"])

        # Display summary text
        m, std = params["mean"], params["std"]
        ax[2].set_title(f'Mean: {m:.2f}\nSTD: {std:.2f}')

        if neuron_i > 20:
            break
        plt.show()

    return all_params

# ==============================================================
# === EXAMPLE USAGE ============================================
# ==============================================================

session_t_start = fr.index[fr.index.get_level_values('session_t') == 40_000]

# Example call (assuming `fr` and `session_t_start` are defined):
all_params = plot_neuron_summary(fr, session_t_start)
# or for a subset:
# plot_neuron_summary(fr, session_t_start, neuron_indices=range(5))


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

def fr_colors(maxval=32):
    """Return (cmap, norm) for firing-rate heatmaps.

    - maxval: maximum FR value (same scale used when building stops).
    - cmap: matplotlib colormap (continuous, 256 colors).
    - norm: Normalize(vmin=0, vmax=maxval) to pass to imshow/Scatter/Colorbar.
    """
    eps = 1e-6
    stops = [
        (0, '#ffffff'),
        ((0 + eps) / maxval, "#e8e8e8"),
        (0.1 / maxval, "#e8e8e8"),
        ((0.1 + eps) / maxval, "#8C6565"),
        (1.0 / maxval, "#8C6565"),
        ((1 + eps) / maxval, "#8C6565"),
        (2.0 / maxval, "#8C6565"),
        ((2 + eps) / maxval, "#963F3F"),
        (4.0 / maxval, "#963F3F"),
        ((4 + eps) / maxval, "#A73131"),
        (8.0 / maxval, "#A73131"),
        ((8 + eps) / maxval, "#D1552C"),
        (16.0 / maxval, "#D1552C"),
        ((16 + eps) / maxval, "#D1922C"),
        (32.0 / maxval, "#D1922C"),
    ]

    # defensive: ensure positions are in [0,1] and sorted
    stops = [(min(max(float(p), 0.0), 1.0), c) for p, c in stops]
    stops = sorted(stops, key=lambda t: t[0])

    # build a continuous colormap from the stop list
    cmap = LinearSegmentedColormap.from_list("fr_cmap", stops, N=256)

    # normalization maps data in [0, maxval] to [0,1] for the colormap
    norm = Normalize(vmin=0.0, vmax=float(maxval))
    return cmap, norm


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from scipy import stats

# ==============================================================
# === COMPUTATION UTILITIES ====================================
# ==============================================================
import matplotlib.colors as mcolors

def compute_distribution_params(unit_hz):
    """Compute firing rate statistics and PMFs for Poisson and Negative Binomial."""
    m = unit_hz.mean()
    std = unit_hz.std()
    var = std ** 2

    # Convert firing rates to discrete counts
    discrete_counts = np.round(unit_hz.values).astype(int)
    discrete_counts = discrete_counts[discrete_counts >= 0]
    lam = discrete_counts.mean()

    # PMFs
    x = np.arange(0, max(10, discrete_counts.max()) + 1)
    pois_pmf = stats.poisson.pmf(x, lam)

    # Negative binomial params
    p = m / var if var > m else 0.999  # prevent division by zero
    r = m * p / (1 - p)
    nb_pmf = stats.nbinom.pmf(x, r, p)

    return {
        "mean": m,
        "std": std,
        "x": x,
        "pois_pmf": pois_pmf,
        "nb_pmf": nb_pmf,
    }

# ==============================================================
# === GRID PLOTTING ============================================
# ==============================================================

def plot_neuron_session_histograms(fr, session_t_start, neuron_indices=None, n_bins=10):
    """
    Plot log(count histograms) for each neuron (rows) and session (columns).
    Each subplot shows histogram of log firing rates and mean/std in title.
    """
    if neuron_indices is None:
        neuron_indices = range(fr.shape[1])
    
    session_names = [f"S{s_id:02}" for s_id, _, _ in session_t_start]
    n_sessions = len(session_names)
    n_neurons = len(neuron_indices)

    fig, axes = plt.subplots(
        nrows=n_neurons, ncols=n_sessions,
        figsize=(1.4 * n_sessions, 2 * n_neurons),
        sharex=True, sharey=True
    )
    
    cmap, norm = fr_colors(maxval=32)
    
    # fig.colorbar(im, ax=ax, label='Firing rate (Hz)')

    # Ensure 2D array of axes
    if n_neurons == 1:
        axes = np.expand_dims(axes, 0)
    if n_sessions == 1:
        axes = np.expand_dims(axes, 1)

    fr_index = fr.index.get_level_values('global_t')

    # Compute start-end time of each session
    session_boundaries = []
    for i, (s_id, _, global_t_start) in enumerate(session_t_start):
        global_t_end = (
            session_t_start[i + 1][2]
            if i + 1 < len(session_t_start)
            else fr_index.max()
        )
        session_boundaries.append((s_id, global_t_start, global_t_end))

    # Iterate neurons × sessions
    for row, neuron_i in enumerate(neuron_indices):
        for col, (s_id, t_start, t_end) in enumerate(session_boundaries):
            ax = axes[row, col]
            unit_hz = fr.iloc[:, neuron_i]
            session_mask = (fr_index >= t_start) & (fr_index < t_end)
            session_data = unit_hz[session_mask]

            if np.all(session_data == 0):
                ax.text(0.5, 0.5, "no spikes", ha='center', va='center', fontsize=8)
                # turn off spines
                [ax.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]
                ax.set_xticks([]); ax.set_yticks([])
                # axis off
                ax.axis('off')
                continue

            params = compute_distribution_params(session_data)

            x = params["x"]
            nb_pmf = params["nb_pmf"]
            pois_pmf = params["pois_pmf"]
            ax.plot(x, nb_pmf, 'b--', lw=2, label='NegBinom')
            ax.plot(np.clip(pois_pmf, 1e-22, None), x, 'r--', lw=2)

            bin_edges = np.arange(0, 301, 25)  # 0–300 Hz in 25 Hz steps
            ax.hist(session_data, bins=bin_edges, color='gray', alpha=0.6, density=True)

            m, std = session_data.mean(), session_data.std()
            # ax.set_title(f"{m:.2f} ± {std:.2f}", fontsize=8)

            if row == n_neurons - 1:
                ax.set_xlabel(session_names[col], fontsize=9)
            if col == 0:
                ax.set_ylabel(f"Neuron {neuron_i}", fontsize=9)
            
            # draw a dot for firing rate
            # get max min x y values
            x_corner, y_corner = 250, 50
            ax.scatter([x_corner], [y_corner], color=cmap(norm(m)), s=300, 
                       edgecolor='k', linewidth=.5, zorder=3)
            ax.text(x_corner-60, y_corner, f"{m:.2}Hz", ha='right', va='center',
                    fontsize=8)
            
            sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
            
            # add thin vertical lines every 100, 200, 300 Hz
            # [ax.axvline(x, linestyle='-', linewidth=.5, color='k', alpha=.2) 
            #  for x in (100,200,300)]
            
            # set ticks at the 25 Hz grid and format as integer labels
            ax.xaxis.set_major_locator(MultipleLocator(25))
            ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))
            plt.setp(ax.xaxis.get_ticklabels(), rotation=0, fontsize=8)
            plt.setp(ax.yaxis.get_ticklabels(), rotation=0, fontsize=8)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.grid(axis='x', linestyle='--', alpha=0.4)
            ax.grid(False, axis='y')
            ax.set_yscale('log')
            
            

    plt.tight_layout()
    # fig.colorbar(sm, ax=ax, label='Firing rate (Hz)')

# Example usage (unchanged)
# set of 5 neurons 
plt.close("all")
for neuron_i in range(0,5,5):
    plot_neuron_session_histograms(fr, session_t_start, neuron_indices=range(neuron_i,neuron_i+5))
    plt.show()

PCA AND ENSEMBLES OVERVIEW
=============================

PCA AND ENSEMBLES OVERVIEW
=============================

PCA AND ENSEMBLES OVERVIEW
=============================

PCA AND ENSEMBLES OVERVIEW
=============================

In [ ]:
# firing rates
fr_raw = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr = fr_raw.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0,1,3, ))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])

# for concat projection
fr = fr.fillna(0)
fr_z = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
fr_z

In [ ]:
pca_loadings_sessions = analytics.get_analytics('SessionPCsProj40ms', session_names=session_names)
pca_loadings_sessions = pca_loadings_sessions.drop(['from_ephys_timestamp', 'to_ephys_timestamp'], axis=1)

pca_components = analytics.get_analytics('ConcatenatedPCs40ms', session_names=session_names)
pca_components = pca_components.drop(['eigenvalues', 'explained_variance'], axis=1)

pca_loadings_sessions

In [ ]:
# project single units on PCs (every column), fr has shape every column is one unity, every row one timepoint
pca_loadings_concat = pd.DataFrame((pca_components.values @ fr_z.T.values).T, 
                                    columns=pca_loadings_sessions.columns)

In [ ]:
plt.close('all')
# Get number of columns
n_cols_data = len(pca_loadings_sessions.columns)

# Calculate grid dimensions (7 columns)
n_cols = 7
n_rows = int(np.ceil(n_cols_data / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1, n_rows * 1), squeeze=True, )
axes = axes.flatten()

for i, col in enumerate(pca_loadings_sessions.columns):
    ax = axes[i]
    dat = pca_loadings_sessions[col]
    dat = np.clip(dat, -10, 10)
    
    ax.hist(dat.values, bins=100)
    # ax.set_yscale('log')
    ax.set_title(col, fontsize=7)
    ax.tick_params(labelleft=False, labelbottom=False)

# Hide unused subplots
for i in range(n_cols_data, len(axes)):
    axes[i].axis('off')


plt.suptitle("PCA projection (Sessionwise)")
plt.tight_layout()
plt.show()






# concatenated PCA projection is much more variable, driven by changes in firingrates over sessions

# Get number of columns
n_cols_data = len(pca_loadings_concat.columns)

# Calculate grid dimensions (7 columns)
n_cols = 7
n_rows = int(np.ceil(n_cols_data / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1, n_rows * 1), squeeze=True)
# Create figure with subplots
axes = axes.flatten()

for i, col in enumerate(pca_loadings_concat.columns):
    ax = axes[i]
    dat = pca_loadings_concat[col]
    dat = np.clip(dat, -10, 10)
    
    ax.hist(dat.values, bins=100)
    # ax.set_yscale('log')
    ax.tick_params(labelleft=False, labelbottom=False)
    ax.set_title(col, fontsize=7)

# Hide unused subplots
for i in range(n_cols_data, len(axes)):
    axes[i].axis('off')

plt.suptitle("PCA projection (concat)")
plt.tight_layout()
# plt.show()

In [ ]:
ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
ensambles



In [ ]:
# sn = '2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min'

ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', 
                                        session_names=session_names).drop("to_ephys_timestamp", axis=1)
# make index instead
ensamble_proj.set_index(['session_id', 'from_ephys_timestamp', ], inplace=True)
ensamble_proj




In [ ]:
plt.close('all')

# Get number of ensembles
n_ensembles = ensamble_proj.shape[1]

# Calculate grid dimensions (7 columns)
n_cols = 7
n_rows = int(np.ceil(n_ensembles / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5), squeeze=False, )
axes = axes.flatten()

for i in range(n_ensembles):
    ax = axes[i]
    ax.hist(np.clip(ensamble_proj.iloc[:, i], -4, 7), bins=100, color='steelblue', alpha=0.7, range=(-4, 7))
    ax.tick_params(labelleft=False)
    ax.set_title(f'Ens {i}', fontsize=7)

# Hide unused subplots
for i in range(n_ensembles, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ensamble vectors are orthogonal 

plt.close('all')

# calculate angles between column vectors
from sklearn.metrics.pairwise import cosine_similarity
cos_sim = cosine_similarity(ensambles.T)
im = plt.imshow(cos_sim, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, label='Cosine Similarity')
plt.title('Cosine Similarity between Ensamble Vectors')
plt.xlabel('Ensamble ID')
plt.ylabel('Ensamble ID')
plt.show()


In [ ]:
# THE ICA rotates PCs significatnly. They remain orthonogal, but are very dissimilar to PCs


# get PCs (inputs to ensemble detection before ICA)
pcs = analytics.get_analytics('ConcatenatedPCs40ms', session_names=session_names).iloc[:, :-2]
pcs

results = np.zeros((ensambles.shape[1], pcs.shape[1]))
for i, ens in enumerate(ensambles.columns):
    cos_sim = cosine_similarity(pcs, ensambles[ens].values.reshape(1, -1))
    results[i, :] = cos_sim[:, 0]
    
# cluster rows (ensambles)
from scipy.cluster.hierarchy import linkage, leaves_list
Z = linkage(results, method='average', metric='euclidean')
row_leaf_order = leaves_list(Z)
print("leaf order for rows:", row_leaf_order)
results = results[row_leaf_order, :]

# same for columns (PCs)
Z = linkage(results.T, method='average', metric='euclidean')
col_leaf_order = leaves_list(Z)
print("leaf order for columns:", col_leaf_order)
results = results[:, col_leaf_order]

plt.close('all')
fig = plt.Figure(figsize=(20, 18))
im = plt.imshow(results, cmap='viridis')
plt.colorbar(im, label='Cosine Similarity', orientation='horizontal')

# axis labels
plt.xticks(ticks=np.arange(pcs.shape[1]), labels=[f'PC {i}' for i in col_leaf_order], rotation=90, fontsize=6)
plt.yticks(ticks=np.arange(ensambles.shape[1]), labels=[f'Ensamble {i}' for i in row_leaf_order], fontsize=6)
plt.title('Cosine Similarity between Ensambles and PCs')



In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(12, 4), width_ratios=(5,1))
ax = ax.flatten()
for i in ensamble_proj.columns:
    # fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(29, 6), width_ratios=(15,1))
    ax[0].set_title(f'All ensembles overlayed')
    # ax[0].set_xlabel('Neurons')
    ax[0].set_ylabel('Weight')
    
    
    # markerline1 = ax[0].stem(np.arange(20), ensambles[i].iloc[:20], label=f'{i}, HP', linefmt='y-', markerfmt='yo', basefmt=" ")
    # markerline2 = ax[0].stem(np.arange(20, ensambles.shape[0]), ensambles[i].iloc[20:], label=f'{i}, mPFC', linefmt='m-', markerfmt='mo', basefmt=" ")
    
    # Draw circles only for high weight
    # thr = .2
    # x = np.arange(ensambles.shape[0])[np.abs(ensambles[i]) > thr]
    # ax[0].scatter(x, ensambles[i][np.abs(ensambles[i]) > thr], s=80, facecolors='none', 
    #               edgecolors='k', linewidths=2, zorder=3, label='Otsu thresh.')

    # lims = (-.45, .45)
    # ax[0].set_ylim(lims)
    # ax[0].legend(fontsize=8)

    # ax[1].hist(np.abs(ensambles[i]), bins=10, label=f'{i} Activity', color='gray', alpha=0.7)
    # ax[1].axvline(x=thr, color='k')
    # ax[1].set_xlabel("abs. weight")
    # ax[1].set_ylabel("Count")

    plt.gca().set_title(f'{i} projection')
    plt.gca().set_xlabel('Time (40ms bins)')
    plt.gca().set_ylabel('Activity')
    # ax[2].set_ylim(-.8,.8)
    
    # real = np.clip(ensamble_proj[i], -5, 5)
    # ax[2].scatter(np.arange(len(real)), real.values, label=f'{i} Activity (25 ms)', color='red', alpha=0.04, s=1)
    
    activity_lims = (-.4, 1.6, )
    
    smoothed = np.clip(ensamble_proj[i], -3, 3).groupby(level='session_id').rolling(window=25*60*15, center=True, min_periods=25*60*8 ).mean().values
    ax[2].plot(smoothed, label=f'{i} Activity (15 min mean)', color='blue', alpha=.3)

    # smoothed = np.clip(ensamble_proj[i], -3, 3).groupby(level='session_id').rolling(window=25*30, center=True, min_periods=1 ).mean().values
    # ax[2].plot(smoothed, label=f'{i} Activity (30 s mean)', color='blue', alpha=.3, linewidth=0.5)

    # if i == 0:
    sample_cnt = 0
    for s_id in ensamble_proj.index.unique(level='session_id'):
        print(f"Next session index: {sample_cnt}, session id: {s_id}")
        # kwargs = {'label': 'next session'} if sample_cnt == 0 else {}
        ax[2].axvline(x=sample_cnt, color='gray', linestyle='--', alpha=0.2,)
        ax[2].annotate(f'S{s_id:02}', xy=(sample_cnt, activity_lims[0]), xytext=(sample_cnt, activity_lims[0]+.1),
                    fontsize=6, color='black')
        sample_cnt += len(ensamble_proj.loc[s_id])
    ax[2].set_ylim(activity_lims)
    ax[2].set_ylabel('Activity')
    # ax[2].legend(fontsize=8)

    # plot the ensemble projection histogram
    # med = np.nanmedian(ensamble_proj[i])
    # ax[3].hist(np.clip(ensamble_proj[i], -1.5, 2.5), bins=50, color='steelblue', alpha=0.7, label=f"Median: {med:.2f}")
    # ax[3].vlines(activity_lims, ymin=0, ymax=ax[3].get_ylim()[1], color='k', linestyles='--', alpha=.3)
    # ax[3].set_ylabel('Count')
    # ax[3].legend(fontsize=6)
    # ax[3].set_xlim(-1.6, 2.6)
    # break
plt.tight_layout()
plt.show()


In [ ]:
for i in ensambles.columns:
    fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(12, 4), width_ratios=(5,1))
    # fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(29, 6), width_ratios=(15,1))
    ax = ax.flatten()
    ax[0].set_title(f'{i} template')
    ax[0].set_xlabel('Neurons')
    ax[0].set_ylabel('Weight')
    
    
    markerline1 = ax[0].stem(np.arange(20), ensambles[i].iloc[:20], label=f'{i}, HP', linefmt='y-', markerfmt='yo', basefmt=" ")
    markerline2 = ax[0].stem(np.arange(20, ensambles.shape[0]), ensambles[i].iloc[20:], label=f'{i}, mPFC', linefmt='m-', markerfmt='mo', basefmt=" ")
    
    # Draw circles only for high weight
    thr = .2
    x = np.arange(ensambles.shape[0])[np.abs(ensambles[i]) > thr]
    ax[0].scatter(x, ensambles[i][np.abs(ensambles[i]) > thr], s=80, facecolors='none', 
                  edgecolors='k', linewidths=2, zorder=3, label='Otsu thresh.')

    lims = (-.45, .45)
    ax[0].set_ylim(lims)
    ax[0].legend(fontsize=8)

    ax[1].hist(np.abs(ensambles[i]), bins=10, label=f'{i} Activity', color='gray', alpha=0.7)
    ax[1].axvline(x=thr, color='k')
    ax[1].set_xlabel("abs. weight")
    ax[1].set_ylabel("Count")

    ax[2].set_title(f'{i} projection')
    ax[2].set_xlabel('Time (40ms bins)')
    ax[2].set_ylabel('Activity')
    # ax[2].set_ylim(-.8,.8)
    
    # real = np.clip(ensamble_proj[i], -5, 5)
    # ax[2].scatter(np.arange(len(real)), real.values, label=f'{i} Activity (25 ms)', color='red', alpha=0.04, s=1)
    
    activity_lims = (-.4, 1.6, )
    
    smoothed = np.clip(ensamble_proj[i], -3, 3).groupby(level='session_id').rolling(window=25*60*15, center=True, min_periods=25*60*8 ).mean().values
    ax[2].plot(smoothed, label=f'{i} Activity (15 min mean)', color='blue', alpha=.8)

    smoothed = np.clip(ensamble_proj[i], -3, 3).groupby(level='session_id').rolling(window=25*30, center=True, min_periods=1 ).mean().values
    ax[2].plot(smoothed, label=f'{i} Activity (30 s mean)', color='blue', alpha=.3, linewidth=0.5)

    sample_cnt = 0
    for s_id in ensamble_proj.index.unique(level='session_id'):
        print(f"Next session index: {sample_cnt}, session id: {s_id}")
        # kwargs = {'label': 'next session'} if sample_cnt == 0 else {}
        ax[2].axvline(x=sample_cnt, color='gray', linestyle='--', alpha=0.2,)
        ax[2].annotate(f'S{s_id:02}', xy=(sample_cnt, activity_lims[0]), xytext=(sample_cnt, activity_lims[0]+.1),
                       fontsize=6, color='black')
        sample_cnt += len(ensamble_proj.loc[s_id])
    ax[2].set_ylim(activity_lims)
    ax[2].legend(fontsize=8)

    # plot the ensemble projection histogram
    med = np.nanmedian(ensamble_proj[i])
    ax[3].hist(np.clip(ensamble_proj[i], -1.5, 2.5), bins=50, color='steelblue', alpha=0.7, label=f"Median: {med:.2f}")
    ax[3].vlines(activity_lims, ymin=0, ymax=ax[3].get_ylim()[1], color='k', linestyles='--', alpha=.3)
    ax[3].set_xlabel('Activity')
    ax[3].set_ylabel('Count')
    ax[3].legend(fontsize=6)
    ax[3].set_xlim(-1.6, 2.6)
    plt.tight_layout()
    plt.show()
    # break


In [ ]:
# project single units on PCs (every column), fr has shape every column is one unity, every row one timepoint
pca_loadings_concat = pd.DataFrame((pca_components.values @ fr_z.T.values).T, 
                                    columns=pca_loadings_sessions.columns)